In [1]:
import os 
from quad_race_env import *
from randomization import *
from stable_baselines3 import PPO
from quadcopter_animation import animation
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def get_log(log_dir):
    event_acc = EventAccumulator(log_dir)
    event_acc.Reload()
    tags = event_acc.Tags()['scalars']
    
    # print(tags)
    # raise Exception('stop')
    log = {'step': [], 'ep_rew_mean': []}
    
    for scalar_event in event_acc.Scalars('rollout/ep_rew_mean'):
        log['step'].append(scalar_event.step)
        log['ep_rew_mean'].append(scalar_event.value)
        
    log = {key: np.array(values) for key, values in log.items()}
    # sub sample in steps of 1e6
    indices = log['step'] % 1e6 == 0
    log = {key: values[indices] for key, values in log.items()}
    # get the highest value and corresponding step
    max_index = np.argmax(log['ep_rew_mean'])
    log['max_ep_rew_mean'] = log['ep_rew_mean'][max_index]
    log['max_step'] = log['step'][max_index]
    return log
    
def get_best_model(log_dir):
    log = get_log(log_dir)
    # print('highest mean episode reward:', log['max_ep_rew_mean'], 'at step:', log['max_step'])
    model_dir = log_dir
    model_dir = model_dir.replace('logs', 'models')
    model_dir = model_dir[:-2]
    path_best_model = model_dir + f'/{log["max_step"]}.zip'
    print('best model:', path_best_model, 'mean_ep_rew:', log['max_ep_rew_mean'])
    return path_best_model, log['max_ep_rew_mean']

env = Quadcopter3DGates(
    num_envs=1,
    randomization=randomization_dummy_30_percent,
    initialize_at_random_gates=True,
    initialize_on_ground=False,
    pause_if_collision=True,
    cam_angle=30.*np.pi/180.,
)

# get latest model from model/ground_exp
def get_latest_model(path):
    models = os.listdir(path)
    # all files end with {number}.zip, get the highest number
    models = sorted(models, key=lambda x: int(x.split('.')[0].split('_')[-1]))
    print(models[-1])
    model = PPO.load(path + '/' + models[-1])
    return model

#  ANIMATION FUNCTION
action_list = []
reward_list = []
def reset_yaw():
    env.reset()
    env.world_states[:, 8] = -np.pi/2 # set yaw to pi

def shift_to_next_gate():
    env.target_gates[0] += 1
    env.target_gates[0] %= len(env.gate_pos)
    env.world_states[:, 0:3] = env.gate_pos[env.target_gates[0]]
    
def animate_policy(model, env, **kwargs):
    env.reset()
    def run():
        actions, _ = model.predict(env.states, deterministic=False)
        action_list.append(actions)
        states, rewards, dones, infos = env.step(actions)
        reward_list.append(rewards)
        print('speed = ', np.mean(np.linalg.norm(states[:, 3:6], axis=1)))
        print('crashed = ', np.sum(dones))
        rate = states[9:12]
        # if any rate is higher 1800 pritn
        if np.any(np.abs(rate) > 1800*np.pi/180):
            print(rate)
            print('rate too high')
        if dones[0]:
            print('out_of_bounds:', infos[0]['out_of_bounds'])
            print('ground_collision:', infos[0]['ground_collision'])
            print('gate_collision:', infos[0]['gate_collision'])
            if infos[0]['out_of_bounds']:
                print(env.world_states[0, 0:3])
                print(env.world_states[0, 9:12])
        
        # print(infos)
        return env.render()
    animation.view(run, gate_pos=env.gate_pos, gate_yaw=env.gate_yaw, reset_func=env.reset, cam_angle=env.cam_angle, **kwargs)

# model, _ = get_best_model('logs/perception_exp/long_oval_good_axis_convention_from_ground_1mgate_0')
# model, _ = get_best_model('logs/perception_exp/long_oval_good_axis_convention_from_ground_1mgate_no_perception_reward_0')
# model, _ = get_best_model('logs/new_perception/IROS_TRACK_G3_G4_0')
model, _ = get_best_model('logs/new_perception/TII_RATM_TRACK_0')

# model = 'models/new_perception/IROS_TRACK_G3_G4/400000000'
# model = 'models/perception_exp/long_oval_good_axis_convention_from_ground_1mgate/10000000'
model = PPO.load(model)

animate_policy(model, env)

python version: 3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:45:18) [GCC 12.3.0]
stable_baselines3 version: 2.3.2
torch version: 2.4.1
cuda available: True
cuda version: 12.4
cudnn version: 90100
device: cuda
test
<function _lambdifygenerated at 0x78e3f0206f80>


/home/robinferede/miniconda3/envs/quad/lib/python3.10/site-packages/stable_baselines3/common/vec_env/base_vec_env.py:77: UserWarning: The `render_mode` attribute is not defined in your environment. It will be set to None.
  warnings.warn("The `render_mode` attribute is not defined in your environment. It will be set to None.")


best model: models/new_perception/TII_RATM_TRACK/239000000.zip mean_ep_rew: 1.1675580739974976


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/robinferede/miniconda3/envs/quad/lib/python3.10/site-packages/cv2/qt/plugins"
QObject::moveToThread: Current thread (0x5ce7424b9400) is not the object's thread (0x5ce75054c7e0).
Cannot move to target thread (0x5ce7424b9400)

QObject::moveToThread: Current thread (0x5ce7424b9400) is not the object's thread (0x5ce75054c7e0).
Cannot move to target thread (0x5ce7424b9400)

QObject::moveToThread: Current thread (0x5ce7424b9400) is not the object's thread (0x5ce75054c7e0).
Cannot move to target thread (0x5ce7424b9400)

QObject::moveToThread: Current thread (0x5ce7424b9400) is not the object's thread (0x5ce75054c7e0).
Cannot move to target thread (0x5ce7424b9400)

QObject::moveToThread: Current thread (0x5ce7424b9400) is not the object's thread (0x5ce75054c7e0).
Cannot move to target thread (0x5ce7424b9400)

QObject::moveToThread: Current thread (0x5ce7424b9400) is not the object's thread (0x5ce75054c7e0).
Cannot move to

speed =  0.44880036
crashed =  0
unpaused
speed =  0.70237094
crashed =  0
speed =  0.9202346
crashed =  0
speed =  1.0891428
crashed =  0
speed =  1.2171592
crashed =  0
speed =  1.3118962
crashed =  0
speed =  1.373376
crashed =  0
speed =  1.4069879
crashed =  0
speed =  1.4176605
crashed =  0
speed =  1.4119912
crashed =  0
speed =  1.3910517
crashed =  0
speed =  1.3624146
crashed =  0
speed =  1.3363845
crashed =  0
speed =  1.3200473
crashed =  0
speed =  1.3120944
crashed =  0
speed =  1.312283
crashed =  0
speed =  1.3216525
crashed =  0
speed =  1.3341018
crashed =  0
speed =  1.3446609
crashed =  0
speed =  1.345804
crashed =  0
speed =  1.3405701
crashed =  0
speed =  1.3307569
crashed =  0
speed =  1.3208942
crashed =  0
speed =  1.3163861
crashed =  0
speed =  1.3178674
crashed =  0
speed =  1.3329278
crashed =  0
speed =  1.3639216
crashed =  0
speed =  1.4107755
crashed =  0
speed =  1.4762125
crashed =  0
speed =  1.5627007
crashed =  0
speed =  1.6665152
crashed =  0
